In [1]:
### 🧩 1. Setup

import requests
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split
from PIL import Image
from io import BytesIO


In [2]:
### 🔑 2. Authenticate with Hugging Face

from huggingface_hub import login
login("YOUR HUGGING FACE TKOEN")  # replace with your token

In [3]:
### 🌐 3. Fetch data using API

import requests
from PIL import Image
import numpy as np
from io import BytesIO

# Use the rows endpoint, not splits
url = "https://datasets-server.huggingface.co/rows?dataset=sashankrudra%2FCarDD&config=default&split=train&offset=0&length=50"
response = requests.get(url)
data = response.json()

images, labels = [], []

for row in data['rows']:   # ✅ now 'rows' exists
    img_url = row['row']['image']['src']
    label = row['row'].get('label', 0)

    img_data = requests.get(img_url).content
    img = Image.open(BytesIO(img_data)).convert("RGB").resize((128, 128))

    images.append(np.array(img, dtype=np.float32) / 255.0)
    labels.append(label)

X = np.array(images, dtype=np.float32)
y = np.array(labels)

print("Images shape:", X.shape)
print("Labels shape:", y.shape)


Images shape: (50, 128, 128, 3)
Labels shape: (50,)


In [4]:
### 🧠 4. Split dataset

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
### 🏗️ 5. Build CNN model

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')  # 6 damage classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


C:\Users\LENOVO\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
### 🚀 6. Train model

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 226ms/step - accuracy: 0.4250 - loss: 1.5958 - val_accuracy: 1.0000 - val_loss: 5.9605e-08
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 1.000

In [7]:
### 📊 7. Evaluate

loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Validation Accuracy: 1.00


In [8]:

### 💾 8. Save model

model.save("car_damage_cnn_api.h5")

In [9]:
# Define class names in the same order you used for training
class_names = ["dent", "scratch", "crack", "glass", "tire", "lamp"]


In [10]:
### 🧠 Optional: Predict new image

test_img = Image.open(r"C:\Users\LENOVO\Downloads\car_damge-224x224.jpeg").resize((128,128))
test_arr = np.expand_dims(np.array(test_img)/255.0, axis=0)
pred = np.argmax(model.predict(test_arr))
print("Predicted damage class:", pred)
pred = np.argmax(model.predict(test_arr))
print("Predicted damage class:", class_names[pred])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Predicted damage class: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicted damage class: dent
